In [ ]:
# pip install notion-client

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

NOTION_TOKEN = os.getenv("NOTION_TOKEN")

In [8]:
from notion_client import Client

notion = Client(auth=NOTION_TOKEN)

In [9]:
# Рекурсивное чтение (если есть вложенные блоки)

def read_blocks(block_id, indent=0):
    blocks = notion.blocks.children.list(block_id=block_id)
    for block in blocks["results"]:
        block_type = block["type"]
        content = block.get(block_type, {})

        if "rich_text" in content:
            text = "".join(t["plain_text"] for t in content["rich_text"])
            print(" " * indent + f"[{block_type}] {text}")

        # Если у блока есть дочерние — читаем их тоже
        if block.get("has_children"):
            read_blocks(block["id"], indent + 2)

In [10]:
PAGE_ID = "33e4c5bfe3c48041b762f66690778e0b"     # ID страницы

read_blocks(PAGE_ID)

  [paragraph] simple text1
[paragraph] Rich text: bold text, italic text, underlined text, link, striked through text, inline code, inline equation
[paragraph] Rich text2: bold and italic text with underline, bold and italic,
[paragraph] colored text (orange)
[paragraph] colored background text
[heading_1] h1
[heading_2] h2
[heading_3] h3
[heading_4] h4
[code] Block with code (language: Python)
[bulleted_list_item] Bulleted list item 1 Level 1
  [paragraph] Just text (Level 1)
  [bulleted_list_item] Level 2
    [paragraph] Just text (Level 2)
[bulleted_list_item] Bulleted list item 2 Level 1
[numbered_list_item] Numbered list item 1 Level 1
  [numbered_list_item] Level 2
  [paragraph] Just text
[numbered_list_item] Numbered list item 2
[to_do] To-do list 1 (checked)
[to_do] To-do list 2
[toggle] Toggle list title
  [paragraph] Toggle list content
[callout] Callout text
[quote] Quote text
[paragraph] Divider below:
[paragraph] Database - Inline below:
[paragraph] Database - Full page be

In [11]:
# --- 1. Получить метаданные страницы (заголовок и т.д.) ---
page = notion.pages.retrieve(page_id=PAGE_ID)
# Извлекаем заголовок
title = page["properties"]["title"]["title"][0]["plain_text"]
print(f"Заголовок страницы: {title}")

# --- 2. Получить содержимое страницы (блоки) ---
blocks = notion.blocks.children.list(block_id=PAGE_ID)

for block in blocks["results"]:
    block_type = block["type"]
    print(block_type)
    # content = block.get(block_type, {})

    # # Текстовые блоки (параграфы, заголовки и т.д.)
    # if "rich_text" in content:
    #     for text in content["rich_text"]:
    #         print(text["plain_text"])

Заголовок страницы: Test
synced_block
paragraph
paragraph
paragraph
paragraph
heading_1
heading_2
heading_3
heading_4
code
equation
bulleted_list_item
bulleted_list_item
numbered_list_item
numbered_list_item
to_do
to_do
toggle
callout
quote
paragraph
divider
child_page
child_page
paragraph
child_database
paragraph
child_database
paragraph
link_to_page
paragraph
paragraph
paragraph
paragraph
paragraph
paragraph
synced_block
paragraph
image
paragraph
bookmark
paragraph
table
paragraph
table_of_contents
paragraph
column_list
paragraph
paragraph


In [12]:
import json

with open("notion_example_page.json", "w", encoding='utf-8') as file:
    json.dump(blocks, file)